In [1]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler
import pandas as pd
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt
from scipy.spatial.distance import cityblock, euclidean
import pickle
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
import os
from umap import UMAP

import os
os.chdir('/ceph/behrens/peter_doohan/goalNav_mFC_refactor/code')

import GridMaze
from GridMaze.analysis.core import get_sessions as gs

In [3]:
def load_all_sessions(subject, maze):
    subject_IDs = [subject]
    maze_name = f"maze_{maze}"

    sessions = gs.get_maze_sessions(
        subject_IDs=subject_IDs,
        maze_names=[maze_name],
        days_on_maze="all",
        with_data=["navigation_df", "navigation_spike_counts_df", "cluster_metrics", "navigation_routes_df"],
        must_have_data=True,
    )
    return sessions
    
def process_session(session, navigation_only=True):
    spk_df = session.navigation_spike_counts_df.merge(session.navigation_df, left_on='time', right_on='time')
    spk_df = spk_df.merge(session.navigation_routes_df[['route_probability']], left_on='time', right_index=True)
    if navigation_only:
        spk_df = spk_df[spk_df.trial_phase == 'navigation']
    return spk_df


def get_train_test_bins(df):
    unique_trials = df.trial_unique_ID.unique()
    train_trials = unique_trials[np.arange(0, len(unique_trials), 2)]
    test_trials = unique_trials[np.arange(1, len(unique_trials), 2)]
    return df[df.trial_unique_ID.isin(train_trials)], df[df.trial_unique_ID.isin(test_trials)]

def preprocess_data(df, time_window=500):
    df['time_bin'] = (df['time'] // (time_window * 0.001)).astype(int)
    spike_count_columns = [col for col in df.columns if col[0] == 'spike_count']
    df = df.dropna(subset=spike_count_columns + [('centroid_position', 'x'), ('centroid_position', 'y'), ('maze_position', 'simple')])
    binned_data = df.groupby(['time_bin'])['spike_count'].mean()
    
    scaler = RobustScaler()
    scaled_data = scaler.fit_transform(binned_data.spike_count.to_numpy())
    binned_data['spike_count'] = scaled_data

    time_bin_to_index = {time_bin: idx for idx, time_bin in enumerate(binned_data.index)}
    # Create a dictionary for each location
    location_data = {}
    for location in df[('maze_position', 'simple')].unique():
        for action in df['cardinal_movement_direction'].unique():
            location_mask = (df[('maze_position', 'simple')] == location) & (df['cardinal_movement_direction'] == action)
            time_bins_for_location = df.loc[location_mask, 'time_bin'].map(time_bin_to_index).dropna().astype(int)
            
            if len(time_bins_for_location) > 0:
                mean_vector = binned_data.iloc[time_bins_for_location].mean(axis=0, skipna=True).values
                mean_x = df.loc[location_mask, ('centroid_position', 'x')].mean()
                mean_y = df.loc[location_mask, ('centroid_position', 'y')].mean()
                location_data[f'{location}_{action}'] = {
                    'mean_vector': mean_vector,
                    'action': action,
                    'x_coord': mean_x,
                    'y_coord': mean_y
                }
            # print(location_data)
            # break
    return location_data

In [46]:
maze = 1
mouse_num = 'm3'
session = 0
time_window = 500
sessions = load_all_sessions(mouse_num, maze)
df = process_session(sessions[session])
location_data = preprocess_data(df, time_window=time_window)

/tmp/ipykernel_937729/2063871630.py:15: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  spk_df = session.navigation_spike_counts_df.merge(session.navigation_df, left_on='time', right_on='time')


In [48]:
df.cardinal_movement_direction.unique()

array(['N', 'W', 'S', 'E'], dtype=object)

In [47]:
location_data['D1-D2_N']

{'mean_vector': array([-2.00000000e-01, -3.83967499e-02,  2.27680341e-01,  1.71784201e-01,
         0.00000000e+00,  5.93476826e-01,  1.10359564e+00,  5.71949602e-01,
         1.42816092e+00,  1.64764204e-01, -1.47612732e-01,  0.00000000e+00,
         2.67803727e-01, -6.64766202e-01,  0.00000000e+00,  1.91570881e-02,
         5.78676179e-02, -7.91011159e-02,  0.00000000e+00,  9.31827190e-01,
         0.00000000e+00,  2.16271475e-01,  6.32463388e-02,  3.62465319e-01,
         2.18036373e-01,  1.47386234e-01,  6.86339523e-01,  7.47582768e-01,
         3.69409128e-03, -8.18029418e-02,  5.05924573e-01,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  5.51909575e-01, -1.47946584e-01,
         4.85000390e-01,  2.25231867e+00,  2.29122839e-01,  0.00000000e+00,
         3.04109881e-01,  5.89080460e-01,  0.00000000e+00,  4.55984939e-01,
        -3.62815940e-02,  4.83154974e-01,  1.51027851e-01,  5.55215098e-01,
        -2.84650447e-01, -3.44827586e-01, -1.88848673e-02,  4.22413793e-0

In [4]:
time_window=500

for mouse_num in ['m2', 'm3', 'm4', 'm6', 'm7', 'm8']: #['MR41', 'MR33', 'MR34', 'MR44']
    for maze in range(1,3):
        sessions = load_all_sessions(mouse_num, maze)
        for i, session in enumerate(sessions):
            day_on_maze = session.day_on_maze
            fname = f"/ceph/behrens/peter_doohan/goalNav_mFC_refactor/code/GridMaze/analysis/xiao_explore/RSA/all/{mouse_num}/maze{maze}"
            os.makedirs(fname, exist_ok=True)
            try:
                df = process_session(session)
                train_df, test_df = get_train_test_bins(df)
                location_data = preprocess_data(train_df, time_window=time_window)
                print(location_data.keys())
                with open(f'{fname}/day{day_on_maze}_part1.pkl', 'wb') as file:
                    pickle.dump(location_data, file)
                location_data_2 = preprocess_data(test_df, time_window=time_window)
                print(location_data_2.keys())
                with open(f'{fname}/day{day_on_maze}_part2.pkl', 'wb') as file:
                    pickle.dump(location_data_2, file)
            except Exception as e:
                print(f"Processing failed: mouse {mouse_num}, session {i}")
                print(e)

clusters.metrics.htsv not found for m2.2022-06-24.maze
clusters.metrics.htsv not found for m2.2022-06-23.maze


/tmp/ipykernel_931004/2063871630.py:15: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  spk_df = session.navigation_spike_counts_df.merge(session.navigation_df, left_on='time', right_on='time')
/tmp/ipykernel_931004/2063871630.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time_bin'] = (df['time'] // (time_window * 0.001)).astype(int)


dict_keys(['C3_N', 'C3_W', 'C3_S', 'C3-C4_N', 'C3-C4_S', 'C4_N', 'C4_S', 'C4-C5_N', 'C4-C5_S', 'C5_N', 'C5_E', 'C5_S', 'C5-D5_E', 'C5-D5_W', 'D5_E', 'D5_W', 'D5_S', 'D5-E5_E', 'D5-E5_W', 'E5_E', 'E5_W', 'E5-F5_E', 'E5-F5_W', 'F5_E', 'F5_W', 'F5_S', 'F5-G5_E', 'F5-G5_W', 'G5_N', 'G5_W', 'G5_S', 'G4-G5_N', 'G4-G5_S', 'G4_N', 'G4_S', 'G3-G4_N', 'G3-G4_S', 'G3_N', 'G3_S', 'G2-G3_N', 'G2-G3_S', 'G2_N', 'G2_W', 'G2_S', 'F2-G2_E', 'F2-G2_W', 'F2_N', 'F2_E', 'F2_W', 'F2_S', 'F1-F2_N', 'F1-F2_S', 'F1_N', 'F2-F3_N', 'F2-F3_S', 'F3_W', 'F3_S', 'E3-F3_E', 'E3-F3_W', 'E3_E', 'E3_W', 'D3-E3_E', 'D3-E3_W', 'D3_N', 'D3_E', 'D3-D4_N', 'D3-D4_S', 'D4_N', 'D4_E', 'D4_S', 'D4-D5_N', 'D4-D5_S', 'E2-F2_E', 'E2-F2_W', 'E2_E', 'E2_W', 'D2-E2_E', 'D2-E2_W', 'D2_E', 'D2_W', 'D2_S', 'C2-D2_E', 'C2-D2_W', 'C2_N', 'C2_E', 'C2_W', 'C2_S', 'B2-C2_E', 'B2-C2_W', 'B2_E', 'B2_W', 'C2-C3_N', 'C2-C3_S', 'B3-C3_E', 'B3-C3_W', 'B3_E', 'B3_W', 'A3-B3_E', 'A3-B3_W', 'A3_N', 'A3_E', 'A3-A4_N', 'A3-A4_S', 'A4_N', 'A4_S', 'A4-A

/tmp/ipykernel_931004/2063871630.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time_bin'] = (df['time'] // (time_window * 0.001)).astype(int)


dict_keys(['E6_W', 'E6_E', 'D6-E6_W', 'D6-E6_E', 'D6_N', 'D6_E', 'D6-D7_N', 'D6-D7_S', 'D7_W', 'D7_S', 'C7-D7_W', 'C7-D7_E', 'C7_E', 'C7_S', 'C6-C7_N', 'C6-C7_S', 'C6_W', 'C6_N', 'C6_S', 'C5-C6_N', 'C5-C6_S', 'C5_N', 'C5_S', 'C4-C5_N', 'C4-C5_S', 'C4_N', 'C4_S', 'C3-C4_N', 'C3-C4_S', 'C3_W', 'C3_N', 'C3_S', 'C2-C3_N', 'C2-C3_S', 'C2_N', 'C2_E', 'C2_S', 'C2-D2_W', 'C2-D2_E', 'D2_W', 'D2_E', 'D2-E2_W', 'D2-E2_E', 'E2_W', 'E2_E', 'E2-F2_W', 'E2-F2_E', 'F2_W', 'F2_N', 'F2_E', 'F2_S', 'F1-F2_N', 'F2-G2_W', 'F2-G2_E', 'G2_W', 'G2_N', 'G2_S', 'G2-G3_N', 'G2-G3_S', 'G3_N', 'G3_S', 'G3-G4_N', 'G3-G4_S', 'G4_N', 'G4_S', 'E6-F6_E', 'F6_N', 'F6_E', 'F6-F7_N', 'F6-F7_S', 'F7_W', 'F7_S', 'E7-F7_W', 'E7-F7_E', 'E7_E', 'F6-G6_E', 'G6_N', 'G6_S', 'G6-G7_N', 'G6-G7_S', 'G7_S', 'G5-G6_S', 'G5_W', 'G5_S', 'G4-G5_N', 'G4-G5_S', 'G1-G2_N', 'G1-G2_S', 'G1_N', 'C1-C2_N', 'C1-C2_S', 'C1_W', 'C1_N', 'B1-C1_W', 'B1-C1_E', 'B1_E', 'F5-G5_W', 'F5_W', 'E5-F5_W', 'E5-F5_E', 'E5_W', 'D5-E5_W', 'D5_W', 'D5_S', 'D4-D5_

...

dict_keys(['A6_N', 'A6_S', 'A6_E', 'A6-A7_N', 'A6-A7_S', 'A7_S', 'A6-B6_E', 'A6-B6_W', 'B6_N', 'B6_W', 'B6-B7_N', 'B6-B7_S', 'B7_S', 'B7_E', 'B7-C7_E', 'B7-C7_W', 'C7_S', 'C7_W', 'C6-C7_N', 'C6-C7_S', 'C6_N', 'C6_S', 'C5-C6_N', 'C5-C6_S', 'C5_N', 'C5_S', 'C5_E', 'C5_W', 'C5-D5_E', 'C5-D5_W', 'D5_N', 'D5_S', 'D5_W', 'D5-D6_N', 'D5-D6_S', 'D6_N', 'D6_S', 'D6_E', 'C4_N', 'C4_S', 'C4_W', 'C3-C4_N', 'C3-C4_S', 'C3_N', 'C3_S', 'A2_N', 'A2_S', 'A2-A3_N', 'A2-A3_S', 'A3_N', 'A3_S', 'A3-A4_N', 'A3-A4_S', 'A4_N', 'A4_S', 'A4-A5_N', 'A4-A5_S', 'A5_N', 'A5_S', 'A5_E', 'A5-A6_N', 'A5-A6_S', 'F3_N', 'F3_S', 'F3_E', 'F3_W', 'F3-G3_E', 'F3-G3_W', 'G3_S', 'G3_W', 'G2-G3_N', 'G2-G3_S', 'G2_N', 'G2_S', 'G1-G2_N', 'G1-G2_S', 'G1_N', 'E3-F3_E', 'E3-F3_W', 'E3_S', 'E3_E', 'E3_W', 'D3-E3_E', 'D3-E3_W', 'D3_N', 'D3_E', 'F1-F2_N', 'F1-F2_S', 'F1_N', 'F2_N', 'F2_S', 'F2-F3_N', 'F2-F3_S', 'F3-F4_N', 'F3-F4_S', 'F4_N', 'F4_S', 'F4-F5_N', 'F4-F5_S', 'F5_N', 'F5_S', 'F5_E', 'F5_W', 'E5-F5_E', 'E5-F5_W', 'E5_S', 'E5

/tmp/ipykernel_931004/2063871630.py:15: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  spk_df = session.navigation_spike_counts_df.merge(session.navigation_df, left_on='time', right_on='time')
/tmp/ipykernel_931004/2063871630.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time_bin'] = (df['time'] // (time_window * 0.001)).astype(int)


dict_keys(['A5_N', 'A5_S', 'A5_E', 'A5-A6_N', 'A5-A6_S', 'A6_N', 'A6_S', 'A6_E', 'A6-A7_N', 'A6-A7_S', 'A7_S', 'A4-A5_N', 'A4-A5_S', 'A4_N', 'A4_S', 'A3-A4_N', 'A3-A4_S', 'A3_N', 'A3_S', 'A3_E', 'A3-B3_E', 'A3-B3_W', 'B3_W', 'A5-B5_E', 'A5-B5_W', 'B5_E', 'B5_W', 'B5-C5_E', 'B5-C5_W', 'C5_N', 'C5_S', 'C5_E', 'C5_W', 'C4-C5_N', 'C4-C5_S', 'C4_N', 'C4_S', 'C4_W', 'C3-C4_N', 'C3-C4_S', 'C3_N', 'C3_S', 'A2-A3_N', 'A2-A3_S', 'D7_S', 'D6-D7_N', 'D6-D7_S', 'D6_N', 'D6_S', 'D6_E', 'D6-E6_E', 'D6-E6_W', 'E6_N', 'E6_W', 'E6-E7_N', 'E6-E7_S', 'E7_S', 'E7_E', 'D5-D6_N', 'D5-D6_S', 'D5_N', 'D5_S', 'D5_W', 'D4-D5_N', 'D4-D5_S', 'D4_N', 'D4_S', 'D3-D4_N', 'D3-D4_S', 'D3_N', 'D3_E', 'D3-E3_E', 'D3-E3_W', 'E3_S', 'E3_E', 'E3_W', 'E3-F3_E', 'E3-F3_W', 'F3_N', 'F3_S', 'F3_E', 'F3_W', 'F2-F3_N', 'F2-F3_S', 'F2_N', 'F2_S', 'F1-F2_N', 'F1-F2_S', 'F1_N', 'C5-D5_E', 'C5-D5_W', 'C2-C3_N', 'C2-C3_S', 'C2_N', 'C2_E', 'C2_W', 'B2-C2_E', 'B2-C2_W', 'B2_S', 'B2_E', 'C2-D2_E', 'C2-D2_W', 'D2_W', 'B1-B2_N', 'B1-B2_S',

/tmp/ipykernel_931004/2063871630.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time_bin'] = (df['time'] // (time_window * 0.001)).astype(int)


dict_keys(['C2_E', 'C2_W', 'C2_N', 'C2-D2_E', 'C2-D2_W', 'D2_W', 'B2-C2_E', 'B2-C2_W', 'B2_E', 'B2_S', 'C2-C3_N', 'C2-C3_S', 'C3_N', 'C3_S', 'C3-C4_N', 'C3-C4_S', 'C4_W', 'C4_N', 'C4_S', 'C4-C5_N', 'C4-C5_S', 'C5_E', 'C5_W', 'C5_N', 'C5_S', 'C5-C6_N', 'C5-C6_S', 'C6_N', 'C6_S', 'C6-C7_N', 'C6-C7_S', 'C7_W', 'C7_S', 'C5-D5_E', 'C5-D5_W', 'D5_W', 'D5_N', 'D5_S', 'D5-D6_N', 'D5-D6_S', 'D6_E', 'D6_N', 'D6_S', 'D6-E6_E', 'D6-E6_W', 'E6_W', 'E6_N', 'E6-E7_N', 'E6-E7_S', 'E7_E', 'E7_S', 'E7-F7_E', 'E7-F7_W', 'F7_E', 'F7_W', 'F7_S', 'F7-G7_E', 'F7-G7_W', 'G7_W', 'D6-D7_N', 'D6-D7_S', 'D7_S', 'F6-F7_N', 'F6-F7_S', 'F6_N', 'F6_S', 'F5-F6_N', 'F5-F6_S', 'F5_E', 'F5_W', 'F5_N', 'F5_S', 'F4-F5_N', 'F4-F5_S', 'F4_N', 'F4_S', 'F3-F4_N', 'F3-F4_S', 'F3_E', 'F3_W', 'F3_N', 'F3_S', 'E3-F3_E', 'E3-F3_W', 'E3_E', 'E3_W', 'E3_S', 'D3-E3_E', 'D3-E3_W', 'D3_E', 'D3_N', 'D3-D4_N', 'D3-D4_S', 'D4_N', 'D4_S', 'D4-D5_N', 'D4-D5_S', 'B1-B2_N', 'B1-B2_S', 'B1_E', 'B1_W', 'B1_N', 'A1-B1_E', 'A1-B1_W', 'A1_E', 'A1_N